In [ ]:
# 因為 ReplyMessageRequest 中的 messages 陣列放了兩個 TextMessage。
# LINE Bot 會依序將 messages 內的每個訊息都回傳給使用者，
# 所以當使用者傳送一則訊息時，Bot 會回覆兩次相同內容。

In [87]:
!pip install Flask pyngrok line-bot-sdk requests --quiet

In [ ]:
ngrok_authtoken = "3ClDmzH22sqWEbRSdze4yo4Zgpf_7ycL9z7EcZyfCYuL92PAR"
line_channel_access_token = "1OQmRLQRLLrqhossKfNHRNPO7oKJBICehSGz0ZmTf/I3JuDjHP2cYZX9L8u/MRrHuO6MmNe+szkjDQSHLoKreAvJxy/k7n0QhuWs/M31wA6meIcAA7nMmcPUsi9VR2SWtKdL3ny/TSnTrMKfBml3CwdB04t89/1O/w1cDnyilFU="
line_channel_secret = "79de232aeab6cf1e0df8c7723791c4bc"

port = 5051

In [89]:
from pyngrok import ngrok
import requests

ngrok.kill()
ngrok.set_auth_token(ngrok_authtoken)

tunnel = ngrok.connect(port)
webhook_url = tunnel.public_url

print("Webhook URL:", webhook_url)

Webhook URL: https://gap-unison-monotone.ngrok-free.dev


In [ ]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token(ngrok_authtoken)

tunnel = ngrok.connect(5051)

webhook_url = tunnel.public_url
print(webhook_url)

In [ ]:
def update_line_webhook(webhook_url):
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {"endpoint": webhook_url}

    res = requests.put(url, headers=headers, json=data)

    print("Webhook update:", res.status_code, res.text)

update_line_webhook(webhook_url)

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

In [ ]:
@app.route("/", methods=["POST"])
def callback():

    signature = request.headers["X-Line-Signature"]
    body = request.get_data(as_text=True)

    print("BODY:", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)

    return "OK"

In [ ]:
@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):

    print("EVENT:", event)

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[
                    TextMessage(text=event.message.text),
                    TextMessage(text=event.message.text)
                ]
            )
        )

In [86]:
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5051)